# c3_90—Publish snapshot to Cloud Storage

### ⚠️ ROI maintainers only. This is not a student notebook.

It writes to an ROI-owned bucket and will throw a permissions error for anyone else. It is in the
repo because it belongs next to the thing it mirrors, not because students should run it.

## The important design decision

**This notebook does not reimplement the transformations.** It fetches
`c3_01_load_explore.ipynb` by raw URL and executes it once per corridor with `CORRIDOR`
overridden, exporting the resulting dataframes to Parquet.

That guarantees the snapshot cannot drift from what students actually get. The moment someone
edits the student notebook, this picks the change up on the next run. A hand-maintained copy of
the same logic would be wrong within a week.

## Why it loads BigQuery tables when all it needs is dataframes

Challenge 2 learned this the expensive way, so we inherit the fix rather than the bug.

The student notebook's validation section runs its queries **against the loaded BigQuery tables**,
not against the dataframes in memory. An earlier version of Challenge 2's publisher stripped the
`load_table` calls out on the reasoning that it only needed dataframes. The checks still ran—
against whatever happened to be sitting in the maintainer's dataset from the last manual run. So
23 checks passed for nine cities while grading one stale copy of a tenth.

A validation suite reading the wrong data is worse than no validation suite, because it is
reassuring. So we load each corridor into a **scratch dataset** (`a4i_econ_publish`, deliberately
not the maintainer's own `a4i_econ`), let validation run against it for real, and **refuse to
export a corridor whose checks did not pass.**

## Why the snapshot exists at all

`c3_01_load_explore.ipynb` pulls live from data.sfgov.org, data.sba.gov, lehd.ces.census.gov and
BigQuery public data. Four external dependencies, on a day when 150 people hit them inside the
same ten minutes. If any one is down or rate-limiting, the notebook fails for the entire room at
once.

`scripts/load.sh` rebuilds every table from this snapshot instead, so a single upstream outage
costs a team five minutes rather than their afternoon.

## Bucket layout

```
gs://class-demo/a4i-2026/challenge-3-micro-grants/<corridor>/<table>/*.parquet
```

`a4i-2026` is the generic top level every challenge shares. The bucket needs
`allUsers:objectViewer` so students can read anonymously from their own projects.

## What gets published

We attempt every corridor listed below and let the student notebook's own validation decide.
A corridor that is too thin fails `businesses: has rows` and is **not** published—which is the
behaviour we want, and it means this list can be optimistic. Third Street yields 36 businesses
and should self-exclude; if it does not, the gate is not working.

In [ ]:
# --- Configuration ---------------------------------------------------------
BUCKET  = "class-demo"
PREFIX  = "a4i-2026/challenge-3-micro-grants"

NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-3-micro-grants/main/notebooks/c3_01_load_explore.ipynb")

# Where the per-corridor tables land so the student notebook's validation has
# something real to query. Deliberately NOT "a4i_econ": that is the dataset a
# maintainer uses when running the student notebook by hand, and this notebook
# truncates every table in here once per corridor.
SCRATCH_DATASET = "a4i_econ_publish"
LOCATION        = "US"

# Every corridor we attempt. The publisher does not decide which are good enough
# - the student notebook's own checks do. Third Street is included ON PURPOSE as
# a negative control: it yields 36 businesses, below the floor of 60, so it must
# FAIL and must not appear in the bucket. A run where Third Street publishes
# means the gate is broken.
CORRIDORS = [
    "North Beach",        # tested end to end, the notebook default
    "Chinatown",
    "Market/Castro",
    "Central Market",
    "Mission Street",
    "Union Street",
    "Parkside Taraval",
    "Geary Boulevard",
    "West Portal",
    "24th St",
    "Third Street",       # negative control - MUST fail
]
EXPECT_FAIL = {"Third Street"}

# table name in BigQuery -> variable name in the student notebook
TABLES = {
    "businesses":          "businesses",
    "employer_blocks":     "blocks",
    "supplies":            "supply_edges",
    "draws_footfall_from": "footfall_edges",
    "borrowed_from":       "loan_edges",
    "tract_demographics":  "tract_demographics",
}
# borrowed_from is legitimately empty in some corridors - a corridor where nobody
# took an SBA loan is a real corridor, not a broken one.
OPTIONAL_TABLES = {"borrowed_from", "employer_blocks"}

import google.auth
credentials, PROJECT_ID = google.auth.default()
print(f"Publishing from project: {PROJECT_ID}")
print(f"Scratch dataset        : {PROJECT_ID}.{SCRATCH_DATASET} ({LOCATION})")
print(f"Corridors              : {len(CORRIDORS)} ({len(EXPECT_FAIL)} negative control)")
print(f"Target                 : gs://{BUCKET}/{PREFIX}/<corridor>/<table>/")

## Fetch the student notebook and extract its code

We take the code cells and drop two kinds we must not execute:

- **`%%bigquery` magics** — `exec` cannot run cell magics at all. Both of ours are display cells,
  so nothing downstream depends on them.
- **Appendix B** — the standalone corridor-viability tester. It is a diagnostic, not part of the
  pipeline, and running it eleven times would re-fetch the city API eleven more times for nothing.
  It carries a marker comment naming this notebook, so the skip survives someone retitling it.

**The `load_table` calls stay.** See the note at the top: stripping them is what let a stale
dataset stand in for nine cities' worth of validation in Challenge 2.

Then we rewrite two lines in the config cell — `CORRIDOR`, and `DATASET` so the loads land in the
scratch dataset rather than the maintainer's own. Both rewrites are located **once**, up front,
and test-fired on a probe before the run starts. A rewrite that silently matches nothing is the
failure mode this notebook is most exposed to, so it is checked rather than assumed.

In [ ]:
import re
import requests
import nbformat

SKIP_MARKER = "c3_90_publish_snapshot.ipynb skips this cell"

nb = nbformat.reads(requests.get(NOTEBOOK_URL, timeout=60).text, as_version=4)

sources, skipped = [], {"magic": 0, "marked": 0}
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source
    if src.lstrip().startswith("%%"):          # exec() cannot run cell magics
        skipped["magic"] += 1
        continue
    if SKIP_MARKER in src:                     # Appendix B and anything else opted out
        skipped["marked"] += 1
        continue
    sources.append(src)                        # load_table calls included, on purpose

print(f"{len(sources)} executable cells; skipped {skipped['magic']} magics, "
      f"{skipped['marked']} marked")

# --- Find the config cell, once, and prove it is unambiguous ----------------
# CORRIDOR_TRACTS is assigned in a later cell and must NOT match: the pattern
# requires '=' immediately after optional whitespace, so 'CORRIDOR_TRACTS =' is
# excluded by construction. Verified below rather than assumed.
CORRIDOR_LINE = re.compile(r'(?m)^CORRIDOR\s*=\s*.+$')
DATASET_LINE  = re.compile(r'(?m)^DATASET\s*=\s*[^\n]+')

config_idx = [i for i, s in enumerate(sources)
              if CORRIDOR_LINE.search(s) and DATASET_LINE.search(s) and "CITY" in s]
if len(config_idx) != 1:
    raise RuntimeError(
        f"Expected exactly one config cell assigning both CORRIDOR and DATASET; found "
        f"{len(config_idx)}: {config_idx}. The student notebook's config cell has changed "
        f"shape and the overrides below would not work. Fix this before publishing anything."
    )
CONFIG_IDX = config_idx[0]

assert not CORRIDOR_LINE.search("CORRIDOR_TRACTS = set()"), \
    "CORRIDOR pattern is too loose - it would rewrite CORRIDOR_TRACTS"


def configure(src, corridor):
    """Rewrite CORRIDOR and DATASET in the config cell. Raises rather than no-ops."""
    src, n = CORRIDOR_LINE.subn(lambda _: f'CORRIDOR = "{corridor}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"CORRIDOR substitution matched {n} lines, expected 1")
    src, n = DATASET_LINE.subn(lambda _: f'DATASET = "{SCRATCH_DATASET}"', src, count=1)
    if n != 1:
        raise RuntimeError(f"DATASET substitution matched {n} lines, expected 1")
    return src


# Prove both substitutions work before we spend twenty minutes finding out they do not.
probe = configure(sources[CONFIG_IDX], "West Portal")
assert 'CORRIDOR = "West Portal"' in probe, "corridor override did not take"
assert f'DATASET = "{SCRATCH_DATASET}"' in probe, "dataset override did not take"
print(f"Config cell is #{CONFIG_IDX}; both overrides test-fired successfully.")

## Build, validate, then export—one corridor at a time

Each corridor runs in its own namespace so a failure in one cannot contaminate the next. We keep
going on failure and report at the end—one bad corridor should not cost you the other ten.

Everything between running the cells and uploading a byte is a gate, and each gate is here
because something got past its absence:

**The corridor actually changed.** `CORRIDOR` is read back out of the executed namespace and
compared to what we asked for. In Challenge 2 a broken substitution published nine copies of the
same city, and every row count looked plausible.

**Better than that: the data says which corridor it is.** The student notebook stamps a `corridor`
column on every business row. We assert every row carries the corridor we asked for. That is a
direct proof rather than an inference — row counts cannot tell two corridors apart, but this can.

**The validation actually passed.** `CHECKS` is read out of the namespace and every entry must be
`True`. We also refuse a run that produced *no* checks, or implausibly few — that means the
validation section did not run, and "no failures" from a suite that never executed is the most
dangerous green there is.

**The validation actually ran against this corridor's data.** `DATASET` is confirmed to be the
scratch dataset. If it still says `a4i_econ`, the override missed and the checks graded somebody's
leftovers.

**Each dataframe is captured the moment it exists**, rather than read out of the namespace at the
end. A later cell that binds the same name to something else used to take a table out of the
snapshot silently.

In [ ]:
import io
import time
import traceback
import pandas as pd
from google.cloud import storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

# The student notebook runs 25 checks today. Demand most of them rather than an
# exact count, so adding a check does not break publishing, but deleting the
# whole section does.
MIN_CHECKS = 20

results = {}

for corridor in CORRIDORS:
    slug = corridor.lower().replace(" ", "-").replace("/", "-")
    print(f"\n{'=' * 68}\n{corridor}   ->  {slug}\n{'=' * 68}")
    t0 = time.time()
    ns = {"__name__": "__main__"}
    captured = {}
    try:
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, corridor)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

            # Grab each table as soon as it appears. Whatever a later cell does
            # to the name afterwards is then none of our business.
            for var in TABLES.values():
                val = ns.get(var)
                if isinstance(val, pd.DataFrame):
                    captured[var] = val

        # --- Gate 1: did the override actually take? -------------------------
        ran_as = ns.get("CORRIDOR")
        if ran_as != corridor:
            raise RuntimeError(
                f"corridor override failed: asked for {corridor!r}, notebook ran as "
                f"{ran_as!r}. Nothing uploaded. Fix configure() before rerunning."
            )
        if ns.get("DATASET") != SCRATCH_DATASET:
            raise RuntimeError(
                f"{corridor} validated against DATASET={ns.get('DATASET')!r}, not "
                f"{SCRATCH_DATASET!r}. Those checks graded the wrong tables."
            )

        # --- Gate 2: does the DATA say it is this corridor? ------------------
        biz = captured.get("businesses")
        if biz is None or not len(biz):
            raise RuntimeError("no businesses dataframe was produced")
        stamped = set(biz["corridor"].astype(str).unique())
        if stamped != {corridor}:
            raise RuntimeError(
                f"businesses carry corridor={stamped}, expected {{{corridor!r}}}. "
                f"The data is not what the config said it would be."
            )

        # --- Gate 3: did the student notebook's own validation pass? ---------
        checks = ns.get("CHECKS")
        if not checks:
            raise RuntimeError(
                "the validation section produced no checks - it did not run. Refusing to "
                "publish something nobody checked."
            )
        if len(checks) < MIN_CHECKS:
            raise RuntimeError(
                f"only {len(checks)} checks ran, expected at least {MIN_CHECKS}. The "
                f"validation section is incomplete."
            )
        failed = [(n, d) for n, ok, d in checks if not ok]
        if failed:
            raise RuntimeError("validation failed: " +
                               "; ".join(f"{n} ({d})" for n, d in failed))

        # --- Export ----------------------------------------------------------
        written = []
        for table, var in TABLES.items():
            df = captured.get(var)
            if df is None or not len(df):
                if table in OPTIONAL_TABLES:
                    print(f"  {table:<22} empty - skipped (optional)")
                    continue
                raise RuntimeError(f"required table {table} was empty")
            buf = io.BytesIO()
            df.to_parquet(buf, index=False)
            buf.seek(0)
            blob = bucket.blob(f"{PREFIX}/{slug}/{table}/data.parquet")
            blob.upload_from_file(buf, content_type="application/octet-stream")
            written.append(f"{table}({len(df):,})")
        print(f"  uploaded: {', '.join(written)}")
        results[corridor] = ("OK", f"{len(biz)} businesses, {time.time() - t0:.0f}s")

    except Exception as exc:                                       # noqa: BLE001
        results[corridor] = ("FAILED", str(exc)[:300])
        print(f"  FAILED: {exc}")
        if corridor not in EXPECT_FAIL:
            traceback.print_exc(limit=3)

print(f"\n{'=' * 68}\nSUMMARY\n{'=' * 68}")
for corridor, (status, detail) in results.items():
    tag = "  (negative control - failing is CORRECT)" if corridor in EXPECT_FAIL else ""
    print(f"{status:<8} {corridor:<20} {detail}{tag}")

# The gate itself is under test. A negative control that publishes means the
# gate is not working, and that is worse than any single bad corridor.
for corridor in EXPECT_FAIL:
    if results.get(corridor, ("", ""))[0] == "OK":
        raise RuntimeError(
            f"{corridor} was expected to FAIL validation and it published instead. "
            f"The publish gate is not working - do not trust this run."
        )
print("\nNegative control behaved correctly: the gate rejects thin corridors.")

## Verify the snapshot is loadable—and is actually the corridor it claims to be

Publishing is not the same as publishing something that works, and something that works is not the
same as something that is right. This cell reads every file back the way `load.sh` will, then asks
four questions of it:

1. **Is it there, and does it have rows?** The floor.
2. **Is this the right corridor?** Every business row must carry the corridor name we asked for.
3. **Does the graph have depth?** The one number that decides whether a cascade query returns
   anything. A corridor whose businesses supply nobody would load perfectly and traverse to
   nothing.
4. **Are the corridors actually different from each other?** We fingerprint each corridor's
   business-id set. Two corridors with an identical fingerprint means the override collapsed.

It also prints the two numbers the README's corridor table needs — businesses after filters and
distinct industries — as a paste-ready markdown table, so those figures come from the published
snapshot rather than from somebody's memory of an earlier run.

In [ ]:
import hashlib

problems = []
fingerprints = {}
readme_rows = []

published = [c for c, (s, _) in results.items() if s == "OK"]
print(f"Verifying {len(published)} published corridors\n")

for corridor in published:
    slug = corridor.lower().replace(" ", "-").replace("/", "-")
    print(f"{corridor}")
    frames = {}
    for table in TABLES:
        blob = bucket.blob(f"{PREFIX}/{slug}/{table}/data.parquet")
        if not blob.exists():
            if table not in OPTIONAL_TABLES:
                problems.append(f"{corridor}: {table} missing")
                print(f"  {table:<22} MISSING")
            continue
        frames[table] = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        print(f"  {table:<22} {len(frames[table]):>6,} rows")

    biz = frames.get("businesses")
    if biz is None or not len(biz):
        problems.append(f"{corridor}: businesses empty")
        continue

    # 2. right corridor?
    stamped = set(biz["corridor"].astype(str).unique())
    if stamped != {corridor}:
        problems.append(f"{corridor}: rows stamped {stamped}")
        print(f"     <-- WRONG CORRIDOR: {stamped}")

    # 3. does the graph have depth?
    sup = frames.get("supplies")
    depth = 0
    if sup is not None and len(sup):
        m = sup.merge(sup, left_on="buyer_id", right_on="supplier_id",
                      suffixes=("", "_2"))
        m = m[m.buyer_id_2 != m.supplier_id]
        depth = m.supplier_id.nunique()
    if depth == 0:
        problems.append(f"{corridor}: no two-hop paths - cascade queries return nothing")
        print(f"     <-- NO GRAPH DEPTH")
    else:
        print(f"     two-hop roots: {depth}")

    # 4. fingerprint
    fingerprints[corridor] = hashlib.sha1(
        "".join(sorted(biz["business_id"].astype(str))).encode()).hexdigest()[:12]

    readme_rows.append((corridor, len(biz), int(biz["naics4"].nunique()), depth))

# 4 (continued): no two corridors may share a fingerprint
seen = {}
for corridor, fp in fingerprints.items():
    if fp in seen:
        problems.append(f"{corridor} and {seen[fp]} are identical (fingerprint {fp})")
    seen[fp] = corridor

print(f"\n{'=' * 68}")
if problems:
    print("PROBLEMS:")
    for p in problems:
        print(f"  - {p}")
else:
    print("All published corridors verified: correct corridor, real depth, all distinct.")
print(f"{'=' * 68}\n")

print("Paste-ready for the README corridor table:\n")
print("| Corridor | After our filters | Industries | Two-hop roots |")
print("|---|---:|---:|---:|")
for corridor, n, inds, depth in sorted(readme_rows, key=lambda r: -r[1]):
    print(f"| **{corridor}** | {n} | {inds} | {depth} |")

print("\nFingerprints (proof no two corridors are copies):")
for corridor, fp in fingerprints.items():
    print(f"  {corridor:<20} {fp}")

## Finally: make the bucket readable, and clean up

`load.sh` reads anonymously from a student's own project, so the objects need
`allUsers:objectViewer`. This is idempotent and safe to re-run.

Then drop the scratch dataset. It has served its purpose and leaving eleven corridors' worth of
tables lying around in a maintainer's project is how somebody later validates against the wrong
thing.

In [ ]:
# Public read on the prefix we just wrote. Run once; harmless to repeat.
print("Run this in Cloud Shell if the bucket is not already public:\n")
print(f"  gcloud storage buckets add-iam-policy-binding gs://{BUCKET} \\")
print(f"      --member=allUsers --role=roles/storage.objectViewer\n")
print("Note: gsutil is deprecated - use gcloud storage.\n")

# Report the bucket's actual IAM rather than testing anonymous HTTP.
#
# An earlier version of this cell fetched the object over public HTTPS and
# reported failure when it 403'd. That was an accurate answer to the wrong
# question: `load.sh` never makes an HTTP request. It uses `gcloud storage ls`
# and `bq load` against gs:// URIs, and BigQuery reads the source object with
# the CREDENTIALS OF THE USER WHO SUBMITTED THE JOB. So allAuthenticatedUsers is
# sufficient - every attendee is signed into a Google account - and allUsers is
# not required.
#
# Do NOT reach for the console's "Authenticated URL"
# (storage.cloud.google.com/...) to fix an access problem. That is a browser
# endpoint: it returns HTTP 200 and an HTML page to a script, so a naive fetch
# looks like it worked until the parse fails. See BLOCKERS.md.
policy = bucket.get_iam_policy(requested_policy_version=3)
readers = {}
for binding in policy.bindings:
    for member in binding.get("members", []):
        if member in ("allUsers", "allAuthenticatedUsers"):
            readers.setdefault(member, []).append(binding["role"])

print("Bucket access relevant to load.sh:")
if not readers:
    print("  NEITHER allUsers NOR allAuthenticatedUsers has a role.")
    print("  Students will NOT be able to run load.sh. Grant one of them:")
    print(f"    gcloud storage buckets add-iam-policy-binding gs://{BUCKET} \\")
    print("        --member=allAuthenticatedUsers --role=roles/storage.objectViewer")
for member, roles in readers.items():
    print(f"  {member}: {', '.join(sorted(set(roles)))}")
if "allAuthenticatedUsers" in readers:
    print("\n  allAuthenticatedUsers is granted, which is what load.sh needs and is")
    print("  the tighter of the two options. Anonymous HTTP will 403, correctly.")

print("\nThe real proof is running `bash scripts/load.sh <corridor>` from a fresh")
print("Skills project. Nothing checked here substitutes for that.")

# Drop the scratch dataset.
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID)
bq.delete_dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}", delete_contents=True, not_found_ok=True)
print(f"\nDropped scratch dataset {SCRATCH_DATASET}.")
print("\nNext: bash scripts/load.sh north-beach  - and run it TWICE.")
print("The second run takes the other branch of the dataset-exists test, which")
print("is the branch that broke Challenge 1.")